# MASS CEO Training Colab

This notebook trains the CEO decision policy for the Multi-Agent Startup Simulator using Hugging Face TRL SFT.

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')

## 1. Clone The Repo

Replace the repo URL before running in a fresh Colab.

In [ ]:
# !git clone <YOUR_REPO_URL>
# %cd Scaler_Hack_Finale

## 2. Install Training Dependencies

In [ ]:
!pip install -q -U transformers datasets peft trl accelerate
!pip uninstall -y torchao

## 3. Generate CEO Training Data

In [ ]:
!python train.py \
  --episodes 100 \
  --horizon 30 \
  --output outputs/trajectories.json \
  --sft-output outputs/ceo_sft.jsonl \
  --preference-output outputs/ceo_preferences.jsonl

## 4. Smoke Test SFT

In [ ]:
!python train_ceo_sft.py \
  --dataset outputs/ceo_sft.jsonl \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --output-dir outputs/models/ceo-sft-smoke \
  --epochs 1 \
  --batch-size 1 \
  --gradient-accumulation-steps 8 \
  --max-steps 10

## 5. Train CEO Policy

In [ ]:
!python train_ceo_sft.py \
  --dataset outputs/ceo_sft.jsonl \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --output-dir outputs/models/ceo-sft \
  --epochs 2 \
  --batch-size 1 \
  --gradient-accumulation-steps 8

## 6. Plot Training Loss

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

state_files = list(Path('outputs/models/ceo-sft').rglob('trainer_state.json'))
state_path = sorted(state_files, key=lambda p: p.stat().st_mtime)[-1]
with open(state_path) as f:
    state = json.load(f)

steps = [row['step'] for row in state['log_history'] if 'loss' in row]
losses = [row['loss'] for row in state['log_history'] if 'loss' in row]
plt.figure(figsize=(8, 5))
plt.plot(steps, losses, marker='o')
plt.xlabel('Training step')
plt.ylabel('Loss')
plt.title('CEO SFT Training Loss')
plt.grid(True)
plt.savefig('loss_curve.png', dpi=160, bbox_inches='tight')
plt.show()

## 7. Evaluate Baseline And Trained CEO

In [ ]:
!python evaluation.py --episodes 20 --horizon 30 --save-dir outputs/eval_baseline
!python evaluation.py --episodes 20 --horizon 30 --agent-mode trained_ceo --save-dir outputs/eval_trained_safety